# 01 数据下载

本 Notebook 完成第一部分所有原始数据的下载，包括：

- **1.1** 10 只自选股票的后复权日度行情（2020-01-01 至今）
- **1.2** 市场指数：沪深 300（000300）+ 中证 500（000905）
- **1.3** 宏观经济指标：CPI 同比增速 + 人民币/美元汇率
- **1.4** 财务指标：10 只股票近 5 个年度的 ROE、净利润率、资产负债率、营业收入增速
- **1.5** 所有下载结果记录至 `download_log.txt`

数据来源：[AKShare](https://akshare.akfamily.xyz/)（开源金融数据接口库）

## 0. 环境准备

In [1]:
import os
import time
import datetime
import pandas as pd
import akshare as ak

print(f"AKShare 版本: {ak.__version__}")
print(f"Pandas 版本: {pd.__version__}")

# 自动创建目录结构（os.makedirs 不手动新建）
BASE = os.path.dirname(os.path.abspath('__file__'))
dirs = [
    'data/stock', 'data/index', 'data/macro',
    'data/finance', 'data/clean', 'data/combined', 'output'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("\n目录结构已就绪:")
for d in dirs:
    print(f"  {d}/")

AKShare 版本: 1.18.60
Pandas 版本: 3.0.3

目录结构已就绪:
  data/stock/
  data/index/
  data/macro/
  data/finance/
  data/clean/
  data/combined/
  output/


## 1.5 下载日志工具函数

每次下载均记录时间戳、状态（SUCCESS/FAILED）、任务名称及数据形状。

In [2]:
LOG_FILE = 'download_log.txt'

def log(status, name, info):
    """写入下载日志"""
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {status:<8} {name:<25} {info}\n'
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(line)
    print(line.strip())

# 写入会话开始标记
with open(LOG_FILE, 'a', encoding='utf-8') as f:
    f.write(f"\n{'='*70}\n")
    f.write(f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] 开始下载会话\n")
    f.write(f"{'='*70}\n")
print("日志工具已初始化")

日志工具已初始化


## 1.1 股票日度行情

下载 10 只股票 2020-01-01 至今的**后复权日度行情**，字段包含日期、开盘价、收盘价、最高价、最低价、成交量、成交额。

接口：`ak.stock_zh_a_hist(adjust='hfq')`（东方财富源）；若超时则切换至腾讯源 `ak.stock_zh_a_daily()`。

In [3]:
STOCKS = {
    '603685': '晨丰科技',
    '603319': '美湖股份',
    '600519': '贵州茅台',
    '601288': '农业银行',
    '601166': '兴业银行',
    '600048': '保利发展',
    '000568': '泸州老窖',
    '002179': '中航光电',
    '300510': '金冠股份',
    '000988': '华工科技',
}

START_DATE = '20200101'
END_DATE = '20260523'
COL_MAP = {
    '日期': 'date', '开盘': 'open', '收盘': 'close',
    '最高': 'high', '最低': 'low', '成交量': 'volume', '成交额': 'amount'
}
# 腾讯源字段映射
TENCENT_COL_MAP = {
    'date': 'date', 'open': 'open', 'close': 'close',
    'high': 'high', 'low': 'low', 'volume': 'volume', 'outstanding_share': 'amount'
}

def get_stock_prefix(code):
    if code.startswith('6'):
        return 'sh'
    else:
        return 'sz'

for code, name in STOCKS.items():
    out_path = f'data/stock/stock_{code}.csv'
    try:
        df = ak.stock_zh_a_hist(
            symbol=code, period='daily',
            start_date=START_DATE, end_date=END_DATE, adjust='hfq'
        )
        df = df.rename(columns=COL_MAP)[['date','open','close','high','low','volume','amount']]
        df.to_csv(out_path, index=False, encoding='utf-8-sig')
        log('SUCCESS', f'stock_{code}({name})', f'shape={df.shape}')
    except Exception as e:
        # 切换腾讯源
        try:
            prefix = get_stock_prefix(code)
            df = ak.stock_zh_a_daily(
                symbol=f'{prefix}{code}',
                start_date=START_DATE, end_date=END_DATE, adjust='hfq'
            )
            df = df.reset_index()
            df = df.rename(columns={'date':'date','open':'open','close':'close',
                                    'high':'high','low':'low','volume':'volume','outstanding_share':'amount'})
            df = df[['date','open','close','high','low','volume','amount']]
            df.to_csv(out_path, index=False, encoding='utf-8-sig')
            log('SUCCESS', f'stock_{code}({name})[腾讯源]', f'shape={df.shape}')
        except Exception as e2:
            log('FAILED ', f'stock_{code}({name})', f'Error: {str(e2)[:60]}')
    time.sleep(0.3)

print("\n股票行情下载完毕")

[2026-05-24 00:24:36] SUCCESS  stock_603685(晨丰科技)[腾讯源]   shape=(1543, 8)


[2026-05-24 00:24:37] SUCCESS  stock_603319(美湖股份)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:38] SUCCESS  stock_600519(贵州茅台)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:39] SUCCESS  stock_601288(农业银行)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:40] SUCCESS  stock_601166(兴业银行)[腾讯源]   shape=(1545, 8)


...

[2026-05-24 00:24:42] SUCCESS  stock_000568(泸州老窖)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:43] SUCCESS  stock_002179(中航光电)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:45] SUCCESS  stock_300510(金冠股份)[腾讯源]   shape=(1545, 8)


[2026-05-24 00:24:45] SUCCESS  stock_000988(华工科技)[腾讯源]   shape=(1545, 8)



股票行情下载完毕


## 1.2 市场指数

下载沪深 300（000300）和中证 500（000905）日度数据。

- **沪深 300**：CAPM 分析的市场基准（rm）
- **中证 500**：自选指数，覆盖中小盘，与本组中小市值个股更相关

接口：`ak.stock_zh_index_hist_csindex()`（中证指数官方源，稳定性优于东方财富）

In [4]:
INDICES = {'000300': '沪深300', '000905': '中证500'}

for code, name in INDICES.items():
    out_path = f'data/index/index_{code}.csv'
    try:
        df = ak.stock_zh_index_hist_csindex(
            symbol=code, start_date=START_DATE, end_date=END_DATE
        )
        # 规范字段名
        col_map = {c: c.lower().replace(' ', '_') for c in df.columns}
        df = df.rename(columns=col_map)
        # 标准化为 date/open/close/high/low/volume
        rename = {}
        for c in df.columns:
            if '日期' in c or 'date' in c.lower(): rename[c] = 'date'
            elif '开盘' in c or 'open' in c.lower(): rename[c] = 'open'
            elif '收盘' in c or 'close' in c.lower(): rename[c] = 'close'
            elif '最高' in c or 'high' in c.lower(): rename[c] = 'high'
            elif '最低' in c or 'low' in c.lower(): rename[c] = 'low'
            elif '成交量' in c or 'volume' in c.lower(): rename[c] = 'volume'
        df = df.rename(columns=rename)
        keep = [c for c in ['date','open','close','high','low','volume'] if c in df.columns]
        df = df[keep]
        df.to_csv(out_path, index=False, encoding='utf-8-sig')
        log('SUCCESS', f'index_{code}({name})', f'shape={df.shape}')
    except Exception as e:
        log('FAILED ', f'index_{code}({name})', f'Error: {str(e)[:60]}')
    time.sleep(0.3)

print("\n指数数据下载完毕")

[2026-05-24 00:24:46] SUCCESS  index_000300(沪深300)       shape=(1546, 6)


[2026-05-24 00:24:47] SUCCESS  index_000905(中证500)       shape=(1546, 6)



指数数据下载完毕


## 1.3 宏观经济指标

### 1.3.1 CPI 同比增速

CPI 反映通货膨胀压力，是货币政策制定的重要参考指标。CPI 上升通常促使央行加息，进而压缩市场估值，对权益资产形成负面压力。

### 1.3.2 人民币/美元汇率

汇率影响出口企业的利润（人民币贬值利好出口）和外资持仓偏好（贬值压力下外资倾向减仓 A 股）。本组军工（中航光电）、电子（华工科技）等行业对汇率变动较为敏感，选此指标有助于分析宏观环境对个股的差异化影响。

In [5]:
# CPI 同比增速
try:
    df_cpi = ak.macro_china_cpi_yearly()
    # 规范列名
    df_cpi.columns = [c.lower().replace(' ', '_') for c in df_cpi.columns]
    rename_cpi = {}
    for c in df_cpi.columns:
        if '日期' in c or 'date' in c: rename_cpi[c] = 'date'
        elif '今值' in c or 'value' in c or 'cpi' in c.lower(): rename_cpi[c] = 'cpi_yoy'
    df_cpi = df_cpi.rename(columns=rename_cpi)
    df_cpi.to_csv('data/macro/macro_cpi.csv', index=False, encoding='utf-8-sig')
    log('SUCCESS', 'macro_cpi', f'shape={df_cpi.shape}')
except Exception as e:
    log('FAILED ', 'macro_cpi', f'Error: {str(e)[:60]}')

# 人民币/美元汇率（中国银行中间价，月度）
try:
    df_boc = ak.currency_boc_safe()
    # 取「美元」列，该列为每100美元对应人民币
    usd_col = [c for c in df_boc.columns if '美元' in c][0]
    date_col = df_boc.columns[0]
    df_fx = df_boc[[date_col, usd_col]].copy()
    df_fx.columns = ['date', 'usd_cny_mid']
    df_fx['usd_cny_mid'] = pd.to_numeric(df_fx['usd_cny_mid'], errors='coerce') / 100
    df_fx = df_fx.dropna()
    # 按月聚合（取月均值）
    df_fx['date'] = pd.to_datetime(df_fx['date'])
    df_fx = df_fx.set_index('date').resample('MS').mean().reset_index()
    df_fx['date'] = df_fx['date'].dt.strftime('%Y-%m-%d')
    df_fx.to_csv('data/macro/macro_exchange_rate.csv', index=False, encoding='utf-8-sig')
    log('SUCCESS', 'macro_exchange_rate', f'shape={df_fx.shape}')
except Exception as e:
    log('FAILED ', 'macro_exchange_rate', f'Error: {str(e)[:60]}')

print("\n宏观数据下载完毕")

[2026-05-24 00:24:50] SUCCESS  macro_cpi                 shape=(477, 5)


[2026-05-24 00:24:56] SUCCESS  macro_exchange_rate       shape=(389, 2)

宏观数据下载完毕


## 1.4 财务指标

获取 10 只股票近 5 个年度（2020–2024）的 4 类财务指标：
- **ROE**（净资产收益率）：衡量股东权益回报能力
- **净利润率**：衡量盈利转化效率
- **资产负债率**：衡量财务杠杆与偿债风险
- **营业收入增速**：衡量成长能力

数据整理为**长格式**（Long format）：每行为一只股票一个年度一个指标的观测，字段为 `code, name, year, indicator, value`。

In [6]:
INDICATOR_MAP = {
    '净资产收益率': 'ROE',
    '净利润率': '净利润率',
    '资产负债率': '资产负债率',
    '营业收入增长率': '营业收入增速',
}
TARGET_YEARS = ['2020', '2021', '2022', '2023', '2024']

all_records = []

for code, name in STOCKS.items():
    try:
        df = ak.stock_financial_analysis_indicator(symbol=code, start_year='2020')
        # 找日期列
        date_col = df.columns[0]
        df[date_col] = df[date_col].astype(str)
        df['year'] = df[date_col].str[:4]
        df = df[df['year'].isin(TARGET_YEARS)]
        # 取年末数据（每年取最后一条）
        df = df.sort_values(date_col)
        df = df.drop_duplicates(subset=['year'], keep='last')

        for src_col, std_name in INDICATOR_MAP.items():
            matched = [c for c in df.columns if src_col in c]
            if not matched:
                continue
            col = matched[0]
            for _, row in df.iterrows():
                val = row[col]
                try:
                    val = float(str(val).replace('%','').replace(',',''))
                except:
                    val = None
                all_records.append({
                    'code': code, 'name': name,
                    'year': row['year'], 'indicator': std_name, 'value': val
                })

        log('SUCCESS', f'finance_{code}({name})', f'years={len(df)}')
    except Exception as e:
        log('FAILED ', f'finance_{code}({name})', f'Error: {str(e)[:60]}')
    time.sleep(0.3)

df_long = pd.DataFrame(all_records)
df_long.to_csv('data/finance/finance_ratios.csv', index=False, encoding='utf-8-sig')
print(f"\n财务长格式数据已保存: shape={df_long.shape}")
print(df_long.head(8))

  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-24 00:24:58] SUCCESS  finance_603685(晨丰科技)      years=5


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-24 00:25:00] SUCCESS  finance_603319(美湖股份)      years=5


  0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-24 00:25:13] SUCCESS  finance_300510(金冠股份)      years=5


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-24 00:25:15] SUCCESS  finance_000988(华工科技)      years=5



财务长格式数据已保存: shape=(150, 5)
     code  name  year indicator   value
0  603685  晨丰科技  2020       ROE  9.4500
1  603685  晨丰科技  2021       ROE  8.6100
2  603685  晨丰科技  2022       ROE -3.6900
3  603685  晨丰科技  2023       ROE  7.1400
4  603685  晨丰科技  2024       ROE  0.9600
5  603685  晨丰科技  2020      净利润率  6.6092
6  603685  晨丰科技  2021      净利润率  5.1143
7  603685  晨丰科技  2022      净利润率 -2.5371


## 下载结果汇总

验证所有文件是否正确生成。

In [7]:
import os

summary = []
for dirpath, category in [
    ('data/stock', '股票行情'),
    ('data/index', '市场指数'),
    ('data/macro', '宏观指标'),
    ('data/finance', '财务数据'),
]:
    for f in sorted(os.listdir(dirpath)):
        if f.endswith('.csv'):
            p = os.path.join(dirpath, f)
            df = pd.read_csv(p, encoding='utf-8-sig', dtype={'code': str})
            summary.append({'类型': category, '文件名': f, '行数': len(df), '列数': len(df.columns)})

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))
print(f"\n下载日志已保存至: {LOG_FILE}")

  类型                           文件名   行数  列数
股票行情              stock_000568.csv 1545   8
股票行情              stock_000988.csv 1545   8
股票行情              stock_002179.csv 1545   8
股票行情              stock_300510.csv 1545   8
股票行情              stock_600048.csv 1545   8
股票行情              stock_600519.csv 1545   8
股票行情              stock_601166.csv 1545   8
股票行情              stock_601288.csv 1545   8
股票行情              stock_603319.csv 1545   8
股票行情              stock_603685.csv 1543   8
市场指数              index_000300.csv 1546   6
市场指数              index_000905.csv 1546   6
宏观指标                 macro_cpi.csv  477   5
宏观指标       macro_exchange_rate.csv  389   2
宏观指标 macro_exchange_rate_daily.csv 1546   3
财务数据            finance_000568.csv   24   5
财务数据            finance_000988.csv   24   5
财务数据            finance_002179.csv   24   5
财务数据            finance_300510.csv   24   5
财务数据            finance_600048.csv   24   5
财务数据            finance_600519.csv   24   5
财务数据            finance_601166.c